## Setup Requirements:

Reading from S3 requires specific dependencies and credentials configuration, especially if running outside of a managed environment like AWS EMR or Databricks.

1. Dependencies: Ensure you have the hadoop-aws and aws-java-sdk packages. You can include them when starting Spark:
   - spark-submit --packages org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262 your_script.py

2. Authentication Options: There are multiple ways to provide AWS credentials when reading from S3:
   - Environment Variables: Set AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY as environment variables, which will be used by the Spark session.
   - IAM Roles (Recommended if running on AWS infrastructure): If running Spark on an AWS service like EMR or EC2, assign an IAM role to
 the instances, and the s3a connector will automatically use those credentials.
   - AWS Credentials File: Use the default AWS credentials file (~/.aws/credentials) that boto3 and 
the AWS CLI use, making it easy to switch between tools.
   - In the Hadoop configuration:
     - hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
     - hadoop_conf.set("fs.s3a.access.key", "YOUR_ACCESS_KEY")
     - hadoop_conf.set("fs.s3a.secret.key", "YOUR_SECRET_KEY")
     - If using temporary credentials, also set the session token:
       - hadoop_conf.set("fs.s3a.session.token", "YOUR_SESSION_TOKEN")
       - hadoop_conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")

The following articles are about Spark performance:
   - https://medium.com/@satrupapanda1/step-by-step-guide-for-reading-data-from-s3-using-pyspark-140a99fb19ba
   - https://medium.com/@sounder.rahul/pyspark-optimization-techniques-for-data-engineers-df5033778709

3. Additional Considerations
   - Cluster Configuration: When reading from S3, ensure that your cluster is configured with enough memory and resources to handle the data volume, especially if the dataset is large.
   - PySpark leverages distributed reads, meaning each Spark worker can read from S3 independently to improve speed.

4. Optimizations:
   - Partitioning: If your data is large, partitioning can significantly speed up the reading process. This is particularly relevant for financial data where you could partition by date or region.
   - Caching: If the same dataset needs to be read multiple times, you can cache it in memory using df.cache(), which can reduce read times.
   - For high-volume data, ensure you use the s3a:// connector rather than the older s3n:// or s3:// (unless on EMR), as it is optimized for larger file sets and better performance.

5. Error Handling
   - When reading data from S3, ensure that the file paths are correct and that the IAM role or credentials used have the necessary permissions to access the bucket.
   - Use try-except blocks or PySpark's built-in DataFrame operations to handle corrupted or missing files gracefully.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ReadDataFromS3") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
    .getOrCreate()

# Read single CSV file
s3_bucket_path = "s3a://bucket/path/to/file.csv"
df = spark.read.csv(s3_bucket_path, header=True, inferSchema=True)

# Read all parquet files inside the directory
df = spark.read.parquet("s3a://bucket/folder/")

# Wildcards: Use glob patterns for specific files
df = spark.read.parquet("s3a://bucket/data/year=2024/*/*.parquet")

# Recursive Discovery: If your files are buried in multiple subdirectories
df = spark.read \
          .option("recursiveFileLookup", "true") \
          .parquet("s3a://bucket/top-level-folder/")

# Show data
df.show()